# **CHƯƠNG 7: LARGE LANGUAGE MODELS**

Notebook này hiện thực hóa các kiến thức nền tảng về Large Language Models được trình bày trong **Chương 7** trong cuốn sách *Speech and Language Processing (Jurafsky & Martin)*.

Bài thực hành này nhằm mục đích minh họa sự thay đổi mô hình (Paradigm Shift) lớn nhất trong lịch sử AI: Sự ra đời của các Mô hình Ngôn ngữ Tự hồi quy (Autoregressive Language Models).

Trong bài này, nhóm xây dựng một **Mini-LLM (Mô hình ngôn ngữ thu nhỏ)** sử dụng kiến trúc lõi **Transformer Decoder** để giải quyết bài toán Phân tích Cảm xúc trên tập dữ liệu Tiếng Việt (VLSP 2018). Thay vì xây dựng một bộ phân loại (Classifier) thông thường, nhóm sẽ áp dụng triết lý của LLM thông qua 3 bước cốt lõi:

1. **Text-to-Text Formatting:** Chuyển đổi toàn bộ dữ liệu huấn luyện thành các chuỗi văn bản tự nhiên.

- *Ví dụ:* `"Review: Quán này ngon. Aspect: Ẩm thực => Sentiment: Tích cực <EOS>"`

2. **Autoregressive Pre-training (Dự đoán từ tiếp theo):** Huấn luyện mô hình không phải để "phân loại", mà chỉ để đọc hiểu và **dự đoán từ tiếp theo** dựa trên ngữ cảnh đứng trước nó.

3. **Prompting (Học trong ngữ cảnh - In-context Learning):** Khi muốn mô hình dự đoán cho một câu mới, ta không dùng hàm `predict()`. Ta sẽ "mớm" (Prompt) cho nó nửa đầu câu: *"Review: Phục vụ rất tệ. Aspect: Dịch vụ => Sentiment:"* Và để mô hình tự động sinh ra (Generate) chữ *"Tiêu cực"*!

## **0. Cài đặt thư viện cần thiết**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import math
import re
import os
from collections import Counter

print(f"PyTorch version: {torch.__version__}")
torch.manual_seed(42)

PyTorch version: 2.11.0+cpu


## **1. BƯỚC 1: ĐỊNH DẠNG DỮ LIỆU THÀNH CHUỖI VĂN BẢN (Text-to-Text)**

Trong kỷ nguyên LLM, mô hình không hiểu khái niệm "Features" và "Labels".
Nó chỉ hiểu các chuỗi văn bản. Do đó, ta cần chuyển đổi bộ dữ liệu VLSP 2018 thành các câu hội thoại/hỏi đáp.

Định dạng huấn luyện (Training Prompt) sẽ là: "Review: <văn bản>. Aspect: <khía cạnh> => Sentiment: <cảm xúc> <EOS>"

Trong đó <EOS> (End of Sentence) giúp LLM biết khi nào nên dừng sinh văn bản.

In [ ]:
polarity_map = {'negative': 'TIEU_CUC', 'neutral': 'TRUNG_TINH', 'positive': 'TICH_CUC'}

def parse_vlsp_for_llm(filepath):
    if not os.path.exists(filepath):
        print(f"[CẢNH BÁO] Không tìm thấy file {filepath}.")
        return []

    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read().strip()

    blocks = re.split(r'\n\s*\n', content)
    formatted_texts = []
    aspect_pattern = re.compile(r'\{([^,]+),\s*(positive|negative|neutral)\}')

    for block in blocks:
        lines = block.split('\n')
        if len(lines) >= 3:
            text = lines[1].strip()
            label_line = lines[2].strip()

            # Với mỗi khía cạnh, tạo ra một câu Prompt huấn luyện độc lập
            for aspect, polarity in aspect_pattern.findall(label_line):
                sentiment = polarity_map[polarity.strip()]
                # Tạo chuỗi văn bản hoàn chỉnh để LLM học thuộc cấu trúc
                prompt_str = f"Review: {text} . Aspect: {aspect.strip()} => Sentiment: {sentiment} <EOS>"
                formatted_texts.append(prompt_str)

    return formatted_texts

In [ ]:
# Tải dữ liệu
train_data = parse_vlsp_for_llm('1-VLSP2018-SA-Restaurant-train (7-3-2018).txt')
test_data = parse_vlsp_for_llm('3-VLSP2018-SA-Restaurant-test (8-3-2018).txt')

if train_data:
    print(f"Số lượng câu văn bản huấn luyện: {len(train_data)}")
    print(f"Ví dụ dữ liệu cho LLM học:\n'{train_data[0]}'\n")

Số lượng câu văn bản huấn luyện: 9297
Ví dụ dữ liệu cho LLM học:
'Review: _ Ảnh chụp từ hôm qua, đi chơi với gia đình và 1 nhà họ hàng đang sống tại Sài Gòn. _ Hôm qua đi ăn trưa muộn, ai cũng đói hết nên lúc có đồ ăn là nhào vô ăn liền, bởi vậy mới quên chụp các phần gọi thêm với nước mắm, chỉ chụp món chính thôi! _ Đói quá nên không biết đánh giá đồ ăn kiểu gì luôn 😅😅😅_ Chọn cái này vì thấy nó lạ với tui. . Aspect: FOOD#STYLE&OPTIONS => Sentiment: TRUNG_TINH <EOS>'



## **2. BƯỚC 2: TOKENIZATION & TẠO TẬP DỮ LIỆU AUTOREGRESSIVE**

Mô hình dự đoán từ tiếp theo cần Input (X) và Target (Y). Khác với các mô hình trước, Target (Y) của LLM chính là Input (X) dịch sang phải 1 vị trí.

Ví dụ chuỗi: A B C D
- X (Input) : A B C
- Y (Target): B C D

In [ ]:
def tokenize(text):
    # Cắt từ, giữ lại các ký tự đặc biệt, dấu chấm và mũi tên =>
    return re.findall(r'<EOS>|=>|\w+|[.]', text.lower())

vocab = {'<PAD>': 0, '<UNK>': 1}
word_counter = Counter()

# Đếm từ vựng
for text in train_data:
    word_counter.update(tokenize(text))

# Xây dựng từ điển (lấy 3000 token phổ biến nhất để mô hình nhẹ)
for word, _ in word_counter.most_common(3000):
    if word not in vocab:
        vocab[word] = len(vocab)

inv_vocab = {v: k for k, v in vocab.items()}
print(f"Kích thước Từ điển (Vocab Size): {len(vocab)} tokens")

Kích thước Từ điển (Vocab Size): 3002 tokens


In [ ]:
MAX_SEQ_LEN = 64

def create_causal_lm_dataset(data):
    X, Y = [], []
    for text in data:
        tokens = tokenize(text)
        token_ids = [vocab.get(w, vocab['<UNK>']) for w in tokens]

        # Cắt hoặc đệm (padding) để câu có độ dài cố định
        if len(token_ids) > MAX_SEQ_LEN + 1:
            token_ids = token_ids[:MAX_SEQ_LEN + 1]
        else:
            token_ids += [vocab['<PAD>']] * (MAX_SEQ_LEN + 1 - len(token_ids))

        # Dịch chuyển (Shift) để tạo Input và Target
        X.append(token_ids[:-1]) # Lấy từ đầu đến sát cuối
        Y.append(token_ids[1:])  # Lấy từ vị trí thứ 2 đến cuối cùng

    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)

# Lấy 1500 mẫu để train nhanh trong bài Demo
subset_train_data = train_data[:1500] if len(train_data) > 1500 else train_data
X_train, Y_train = create_causal_lm_dataset(subset_train_data)
print(f"Kích thước X_train (Input): {X_train.shape} | Y_train (Target): {Y_train.shape}")

Kích thước X_train (Input): torch.Size([1500, 64]) | Y_train (Target): torch.Size([1500, 64])


## **3. BƯỚC 3: XÂY DỰNG MINI-LLM BẰNG TRANSFORMER DECODER**

Đây là trái tim của Chương 7. Mô hình sử dụng kiến trúc Transformer. Điểm quan trọng nhất là **Causal Mask (Mặt nạ nhân quả)**: Nó là một ma trận tam giác trên, che đi các từ trong tương lai, ép mô hình ở vị trí t chỉ được phép tập trung (attention) vào các vị trí từ 1 đến t.

In [ ]:
class MiniLLM_Transformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, max_seq_len=MAX_SEQ_LEN):
        super(MiniLLM_Transformer, self).__init__()
        self.d_model = d_model

        # 1. Embedding từ vựng và Embedding vị trí (Positional Encoding)
        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)

        # 2. Khối Transformer Encoder (Nhưng hoạt động như Decoder nhờ Causal Mask)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=d_model*4,
                                                   batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 3. Lớp Linear xuất ra xác suất cho toàn bộ Từ điển
        self.fc_out = nn.Linear(d_model, vocab_size)

    def generate_square_subsequent_mask(self, sz):
        # Tạo ma trận mask tam giác (Causal Mask)
        # Các giá trị trên đường chéo bị gán -inf (âm vô cùng) để model không "nhìn" được tương lai
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def forward(self, x):
        seq_len = x.size(1)

        # Tạo tensor chứa vị trí [0, 1, 2, ..., seq_len-1]
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)

        # Cộng vector Token và vector Vị trí
        x_emb = self.token_emb(x) + self.pos_emb(positions)
        x_emb = x_emb * math.sqrt(self.d_model)

        # Tạo Causal Mask
        causal_mask = self.generate_square_subsequent_mask(seq_len).to(x.device)

        # Đưa qua Transformer (Tham số is_causal hỗ trợ tối ưu trên các bản PyTorch mới)
        out = self.transformer(x_emb, mask=causal_mask, is_causal=True)

        # Tính toán Logits (Xác suất từ vựng)
        logits = self.fc_out(out)
        return logits

In [ ]:
# Khởi tạo mô hình
llm_model = MiniLLM_Transformer(vocab_size=len(vocab))
criterion = nn.CrossEntropyLoss(ignore_index=0) # Bỏ qua tính Loss cho token <PAD>
optimizer = optim.Adam(llm_model.parameters(), lr=0.001)

## **4. BƯỚC 4: HUẤN LUYỆN LLM (PRE-TRAINING)**

Ta dạy LLM cách đọc và hiểu cấu trúc ngôn ngữ của bộ dữ liệu VLSP bằng cách bắt nó liên tục dự đoán từ tiếp theo trong hàng ngàn câu review.

In [ ]:
epochs = 15
batch_size = 64

if len(X_train) > 0:
    print("\n--- BẮT ĐẦU HUẤN LUYỆN MINI-LLM (AUTOREGRESSIVE PRE-TRAINING) ---")
    for epoch in range(epochs):
        llm_model.train()
        total_loss = 0

        for i in range(0, len(X_train), batch_size):
            b_x = X_train[i : i+batch_size]
            b_y = Y_train[i : i+batch_size]

            optimizer.zero_grad()
            logits = llm_model(b_x)

            # Reshape logits và targets để đưa vào hàm CrossEntropyLoss
            # Logits shape: (batch_size * seq_len, vocab_size)
            # Targets shape: (batch_size * seq_len)
            loss = criterion(logits.view(-1, len(vocab)), b_y.view(-1))

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / (len(X_train) / batch_size)
        print(f"Epoch {epoch+1:2d} | Causal LM Loss: {avg_loss:.4f} | Perplexity: {math.exp(min(avg_loss, 20)):.2f}")


--- BẮT ĐẦU HUẤN LUYỆN MINI-LLM (AUTOREGRESSIVE PRE-TRAINING) ---
Epoch  1 | Causal LM Loss: 7.1946 | Perplexity: 1332.23
Epoch  2 | Causal LM Loss: 6.3932 | Perplexity: 597.74
Epoch  3 | Causal LM Loss: 6.2060 | Perplexity: 495.73
Epoch  4 | Causal LM Loss: 5.9043 | Perplexity: 366.62
Epoch  5 | Causal LM Loss: 5.5929 | Perplexity: 268.51
Epoch  6 | Causal LM Loss: 5.2978 | Perplexity: 199.90
Epoch  7 | Causal LM Loss: 5.0194 | Perplexity: 151.31
Epoch  8 | Causal LM Loss: 4.7539 | Perplexity: 116.04
Epoch  9 | Causal LM Loss: 4.4983 | Perplexity: 89.87
Epoch 10 | Causal LM Loss: 4.2437 | Perplexity: 69.66
Epoch 11 | Causal LM Loss: 3.9980 | Perplexity: 54.49
Epoch 12 | Causal LM Loss: 3.7644 | Perplexity: 43.14
Epoch 13 | Causal LM Loss: 3.5282 | Perplexity: 34.06
Epoch 14 | Causal LM Loss: 3.3036 | Perplexity: 27.21
Epoch 15 | Causal LM Loss: 3.1009 | Perplexity: 22.22


## **5. BƯỚC 5: SINH VĂN BẢN (GENERATION) & PROMPTING**

Đây là phần chứng minh khái niệm quan trọng nhất của Chương 7: **In-context Learning**.
- Mô hình đã học được quy luật: Sau chữ "=> Sentiment:" chắc chắn sẽ là từ chỉ cảm xúc.
- Do đó, khi sử dụng, ta không phân loại, ta chỉ **mớm (Prompt)** một nửa câu và bắt mô hình dùng kỹ thuật **Greedy Decoding** để viết nốt nửa còn lại!

In [ ]:
def generate_text_from_prompt(model, prompt_text, max_new_tokens=5):
    model.eval()

    # Tokenize Prompt
    tokens = tokenize(prompt_text.lower())
    input_ids = [vocab.get(w, vocab['<UNK>']) for w in tokens]

    generated_tokens = []

    with torch.no_grad():
        for _ in range(max_new_tokens):
            x_tensor = torch.tensor([input_ids], dtype=torch.long)

            # Dự đoán toàn bộ chuỗi
            logits = model(x_tensor)

            # Lấy Logits của TỪ CUỐI CÙNG trong chuỗi hiện tại
            next_token_logits = logits[0, -1, :]

            # Lấy Token có xác suất cao nhất (Greedy)
            predicted_id = torch.argmax(next_token_logits).item()

            if predicted_id == vocab.get('<EOS>', -1):
                break # Dừng sinh văn bản nếu gặp token EOS

            generated_tokens.append(inv_vocab[predicted_id])
            input_ids.append(predicted_id) # Nối từ mới vào Input để dự đoán tiếp

    return " ".join(generated_tokens)

In [ ]:
# DEMO PROMPTING CHO BÀI TOÁN ABSA
print("\n--- DEMO: SỬ DỤNG PROMPTING ĐỂ PHÂN LOẠI CẢM XÚC ---")
# Lưu ý: Các câu này ta mớm (Prompt) đúng format mà mô hình đã học ở Bước 1.
prompts = [
    "Review: đồ ăn ở đây ngon lắm . Aspect: food#quality => Sentiment:",
    "Review: giá cả quá đắt đỏ so với mặt bằng chung . Aspect: food#prices => Sentiment:",
    "Review: nhân viên phục vụ rất chậm chạp . Aspect: service#general => Sentiment:"
]

for prompt in prompts:
    # Mô hình sẽ tự động sinh ra (Generate) nhãn cảm xúc
    response = generate_text_from_prompt(llm_model, prompt, max_new_tokens=3)

    print(f"Câu hỏi (Prompt) : '{prompt}'")
    print(f"LLM Trả lời      : [{response.upper()}]")
    print("-" * 75)


--- DEMO: SỬ DỤNG PROMPTING ĐỂ PHÂN LOẠI CẢM XÚC ---
Câu hỏi (Prompt) : 'Review: đồ ăn ở đây ngon lắm . Aspect: food#quality => Sentiment:'
LLM Trả lời      : [TICH_CUC EOS CAY]
---------------------------------------------------------------------------
Câu hỏi (Prompt) : 'Review: giá cả quá đắt đỏ so với mặt bằng chung . Aspect: food#prices => Sentiment:'
LLM Trả lời      : [TICH_CUC EOS CÓ]
---------------------------------------------------------------------------
Câu hỏi (Prompt) : 'Review: nhân viên phục vụ rất chậm chạp . Aspect: service#general => Sentiment:'
LLM Trả lời      : [TICH_CUC EOS CẢ]
---------------------------------------------------------------------------
